# Data pipeline

## Imports

In [2]:
# Imports
import pandas as pd
from pathlib import Path
import re

## Constants

In [3]:
MODULE_1 = [
    "GH 1",
    "GH 2",
    "GH 3",
    "GH 4",
    "GH 5",
    "GH 6",
    "GH 7",
    "GH 8",
    "GH 9",
    "GH 10",
    "GH 11",
    "GH 12",
    "GH 13",
    "GH 14",
    "GH 15",
    "GH 17",
    "GH 19",
    "GH 21",
    "GH 22",
]

MODULE_2 = [
    "GH 24",
    "GH 25",
    "GH 27",
    "GH 28",
    "GH 29",
    "GH 30",
    "GH 31",
    "GH 32",
    "GH 33",
    "GH 34",
    "GHT",
]

MODULE_3 = ["IF 1", "IF 2", "IF 3", "IF 4", "IF 5", "IF 6", "IFT"]

MODULE_6 = ["AN 1", "AN 2", "AN 3", "AN 4", "AN 5", "AN 6", "AN 7"]

LESSONS_TO_KEEP = MODULE_1 + MODULE_2 + MODULE_3

## Convert raw data from .xlsx files to .parquet

You only have to run this code cell once. Once you have the raw data in .parquet files, you can skip to the next section.

Converting raw .xlsx files to .parquet ensures consistent schema enforcement, faster reads and writes, and significant file size reduction due to Parquet's columnar storage and built-in compression.

### Constants

In [ ]:
# Config
RAW_DATA_XLSX = Path("../data/01_raw/xlsx")
RAW_DATA_PARQUET = Path("../data/01_raw/parquet")
RAW_DATA_PARQUET.mkdir(parents=True, exist_ok=True)

# Schema definition
STRING_COLS = [
    "STUDENT_ID",
    "STUDENT_FIRST_NAME",
    "STUDENT_LAST_NAME",
    "STUDENT_DISPLAY_NAME",
    "STUDENT_TITLE",
    "STUDENT_RANK",
    "STUDENT_RANK_ABBREV",
    "COURSE",
    "CLASS_TEMP_DS_ID",
    "CLASS",
    "CLASS_DS_ID",
    "MODULE_NAME",
    "MODULE_DS_ID",
    "MODULE_TEMP_NAME",
    "MODULE_TEMP_DS_ID",
    "LESSON_NUMBER",
    "LESSON_DS_ID",
    "LESSON_DESCRIPTION",
    "PLAN_DISPLAY_ID",
    "LESSON_TEMPLATE_COMMENTS",
    "LESSON_TEMP_NAME",
    "LESSON_TEMP_DS_ID",
    "PLAN_TEMPLATE_DURATION_TYPE",
    "PLAN_TEMPLATE_DS_ID",
    "PLAN_TYPE_ID",
    "SUB_PLAN_TYPE_ID",
    "PLAN_CATEGORY_ID",
    "ABORT_REASON",
    "ABORT_CODE",
    "DUTY_STATUS_DESCRIPTION",
    "WEATHER_CODE_DESCRIPTION",
    "SOLO_SHEET_MAN_NOT",
    "INSTRUCTOR_ID",
    "INSTRUCTOR_FIRST_NAME",
    "INSTRUCTOR_LAST_NAME",
    "INSTRUCTOR_DISPLAY_NAME",
    "INSTRUCTOR_TITLE",
    "INSTRUCTOR_RANK",
    "INSTRUCTOR_RANK_ABBREV",
    "EMP_EVAL_COMMENTS",
    "EVAL_ACK_BY_EMPLOYEE_DS_ID",
    "EMP_ACK_BY_EMPLOYEE_DS_ID",
    "OBJECTIVE_ID",
    "OBJECTIVE_DESCRIPTION",
    "OBJECTIVE_ACCEPTABLE_SCORE_DSC",
    "OBJECTIVE_EXPECTED_SCORE_DSC",
    "OBJECTIVE_COMMENTS",
]

FLOAT_COLS = [
    "PLAN_TEMPLATE_DURATION",
    "ACTUAL_FLIGHT_HOURS",
    "AVERAGE_SCORE",
    "ADJUSTED_SCORE",
    "EMP_EVAL_SCORE",
    "EMP_EVAL_RANKING",
    "OBJECTIVE_RAW_SCORE",
    "OBJECTIVE_WEIGHTED_SCORE",
    "OBJECTIVE_SCORE_WEIGHT",
    "OBJECTIVE_ACCEPTABLE_SCORE",
    "OBJECTIVE_EXPECTED_SCORE",
]

BOOLEAN_COLS = [
    "EVALUATOR_ACKNOWLEDGED",
    "EMPLOYEE_ACKNOWLEDGED",
    "OBJECTIVE_CRITICAL",
]

DATE_COLS = [
    "COURSE_START_DATE",
    "COURSE_END_DATE",
    "COURSE_ACTUAL_START_DATE",
    "COURSE_ACTUAL_END_DATE",
    "LESSON_START_DATE",
    "LESSON_END_DATE",
    "LESSON_ACTUAL_START_DATE",
    "LESSON_ACTUAL_END_DATE",
    "TAKE_OFF_TIME",
    "LANDING_TIME",
    "EVALUATOR_ACKNOWLEDGED_DATE",
    "EMPLOYEE_ACKNOWLEDGED_DATE",
]

ALL_COLS = set(STRING_COLS) | set(FLOAT_COLS) | set(BOOLEAN_COLS) | set(DATE_COLS)

REQUIRED_COLS = {
    "STUDENT_ID",
    "CLASS",
    "LESSON_NUMBER",
    "DUTY_STATUS_DESCRIPTION",
    "EMP_EVAL_SCORE",
    "ADJUSTED_SCORE",
    "EMP_EVAL_COMMENTS",
}

### Helper functions

In [ ]:
def _to_bool_nullable(series: pd.Series) -> pd.Series:
    """Convert messy boolean-like values into pandas nullable boolean."""
    s = series.astype("string").str.strip().str.lower()

    mapped = s.map(
        {
            "true": True,
            "t": True,
            "yes": True,
            "y": True,
            "1": True,
            "false": False,
            "f": False,
            "no": False,
            "n": False,
            "0": False,
        }
    )

    return mapped.astype("boolean")


def _coerce_datetime(series: pd.Series) -> pd.Series:
    """Safely coerce mixed datetime/time/string into datetime."""
    return pd.to_datetime(series.astype("string"), errors="coerce")


def _add_missing_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure all expected columns exist with correct nullable defaults."""
    n = len(df)
    missing = [c for c in ALL_COLS if c not in df.columns]

    if missing:
        print(f"[WARN] Adding missing columns: {missing}")

    for c in missing:
        if c in STRING_COLS:
            df[c] = pd.Series([pd.NA] * n, dtype="string")
        elif c in FLOAT_COLS:
            df[c] = pd.Series([pd.NA] * n, dtype="Float64")
        elif c in BOOLEAN_COLS:
            df[c] = pd.Series([pd.NA] * n, dtype="boolean")
        elif c in DATE_COLS:
            df[c] = pd.Series([pd.NaT] * n, dtype="datetime64[us]")

    return df


def enforce_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Enforce final schema after raw ingestion.

    - Adds missing optional columns
    - Validates required columns
    - Coerces types safely
    """
    df = df.copy()

    # ---- Required column check ----
    missing_required = REQUIRED_COLS - set(df.columns)
    if missing_required:
        raise ValueError(f"Missing REQUIRED columns: {missing_required}")

    # ---- Add missing optional columns ----
    df = _add_missing_columns(df)

    # ---- Type coercion ----
    for c in STRING_COLS:
        df[c] = df[c].astype("string")

    for c in FLOAT_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Float64")

    for c in BOOLEAN_COLS:
        df[c] = _to_bool_nullable(df[c])

    for c in DATE_COLS:
        df[c] = _coerce_datetime(df[c])

    return df


def read_raw_excel(path: Path) -> pd.DataFrame:
    """
    Read raw Excel file with minimal assumptions.
    All columns are initially read as strings to avoid type issues.
    """
    df = pd.read_excel(
        path,
        engine="calamine",
        dtype="string",  # fully permissive
    )

    return df


def process_file(path: Path) -> pd.DataFrame:
    """
    End-to-end processing for a single file:
    read -> enforce schema
    """
    print(f"\n[INFO] Processing {path.name}")

    df = read_raw_excel(path)
    df = enforce_schema(df)

    return df

### Code

In [ ]:
for path in sorted(RAW_DATA_XLSX.glob("*.xlsx")):
    try:
        df = process_file(path)

        # Save individual parquet (useful for debugging)
        out_path = RAW_DATA_PARQUET / f"{path.stem}.parquet"
        df.to_parquet(out_path, index=False)

    except Exception as e:
        print(f"[ERROR] Failed on {path.name}: {e}")

## Initial cleaning of raw data

### Concat into singular df

In [4]:
# Concatenate all data into a singular df
all_dfs = []
for file in Path("../data/01_raw/parquet").glob("*.parquet"):
    print(file.stem)
    df = pd.read_parquet(file)
    all_dfs.append(df)
df_all = pd.concat(all_dfs, ignore_index=True)

NEST(170-179)
NEST(23-1-24 to 2-4-24)
NEST(180-189)
NEST(160-169)
NEST(1-4-25 to 1-5-25)
NEST(150-159)
NEST(1-5-25 to 29-5-25)
NEST(3-4-24 to 18-8-24)
NEST(3-6-25 to 30-6-25)
NEST(4-3-25 to 31-3-25)
LM_BWC201to205_data_pulled_as_of_end_Jan2026
JULY 25
NEST(1-11-23 to 23-1-24)
LM_BWC206to207_data_pulled_as_of_end_Jan2026
NEST(19-8-24 to 4-3-25)
NEST(190-197)_fixedup


### Drop duplicate rows

In [5]:
# Drop duplicate rows
df = df_all.drop_duplicates().reset_index(drop=True)
print(f"{len(df_all) - len(df)} duplicate rows dropped. {len(df)} rows remain.")

68349 duplicate rows dropped. 1239196 rows remain.


### Drop rows with null values in `STUDENT_ID`

In [6]:
# Drop rows with NaN values in STUDENT_ID
bef = len(df)
df = df.dropna(subset="STUDENT_ID").reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows with null values in `STUDENT_ID` dropped. {aft} rows remain.")

393 rows with null values in `STUDENT_ID` dropped. 1238803 rows remain.


### Drop rows with null values in `EMP_EVAL_SCORE`

In [7]:
bef = len(df)
df = df.dropna(subset="EMP_EVAL_SCORE").reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows with null values in `EMP_EVAL_SCORE` dropped. {aft} rows remain.")

35376 rows with null values in `EMP_EVAL_SCORE` dropped. 1203427 rows remain.


### Drop rows with null values in `LESSON_NUMBER`

In [8]:
bef = len(df)
df = df.dropna(subset="LESSON_NUMBER").reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows with null values in `LESSON_NUMBER` dropped. {aft} rows remain.")

0 rows with null values in `LESSON_NUMBER` dropped. 1203427 rows remain.


### Clean `CLASS` column

In [9]:
assert df["CLASS"].isnull().sum() == 0

In [10]:
# Clean class column - replace weird values found by manual inspection
df["CLASS"] = df["CLASS"].str.replace("192B BWC-B", "192 BWC-B", regex=False)
df["CLASS"] = df["CLASS"].str.replace("202 BWC-B - A", "202 BWC-B", regex=False)

# Remove trailing whitespace
df["CLASS"] = df["CLASS"].str.strip()

### Keep only rows belonging to BWC pilot trainees

In [11]:
# Value in CLASS must follow "XXX BWC-B" format
CLASS_OK_PATTERN = re.compile(r"^\d+\s+BWC-B$")

df_bwc = df.loc[
    df["CLASS"].astype("string").str.strip().str.match(CLASS_OK_PATTERN, na=False)
].reset_index(drop=True)

print(f"{len(df) - len(df_bwc)} rows that are NOT from BWC pilot trainees dropped. These rows belong to BWC WSO trainees, or BWC-F trainees.")

98991 rows that are NOT from BWC pilot trainees dropped. These rows belong to BWC WSO trainees, or BWC-F trainees.


### Create `bwc_batch` column

In [12]:
# Extract numeric prefix at start of CLASS (e.g., "199" from "199 BWC-B")
df_bwc["bwc_batch"] = (
    df_bwc["CLASS"]
    .astype(str)
    .str.extract(r"^(\d+)", expand=False)  # capture leading digits only
    .astype("float")                       # convert to numeric type
    .astype("Int64")                       # optional: use pandas nullable int type
)

### Clean `LESSON_NUMBER` column

In [13]:
def clean_lesson_numbers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize LESSON_NUMBER strings.

    Rules:
    - drop rows where LESSON_NUMBER is NaN
    - remove leading batch prefixes like '150-' or '150- '
    - canonicalize common aliases
    - normalize suffix terms
    - insert spaces between letters and digits
    - collapse whitespace and trim
    """
    if "LESSON_NUMBER" not in df.columns:
        raise KeyError("Expected column 'LESSON_NUMBER' not found.")

    out_df = df.copy()
    out_df = out_df.dropna(subset=["LESSON_NUMBER"]).copy()

    s = out_df["LESSON_NUMBER"].astype("string").str.upper().str.strip()

    # Remove leading batch prefixes like:
    # 152-GH27R, 153- GH 1, 156 GH40FL
    s = s.str.replace(r"^\d+\s*-\s*", "", regex=True)
    s = s.str.replace(r"^\d+\s+", "", regex=True)

    # Remove leading MS prefixes like:
    # MS - GH 1, MS -GH 1, MS-GH 1
    s = s.str.replace(r"^MS\s*-\s*", "", regex=True)

    # Normalize known raw variants
    s = s.str.replace(r"\bG\s*/\s*S\b", "GS", regex=True)
    s = s.str.replace(r"\bFLEX\b", "FL", regex=True)

    # Canonicalize CHECK variants
    s = s.str.replace(r"\bCHECK\b", "CHK", regex=True)
    # GH 9 was originally PROG CHK
    s = s.str.replace(r"\bPROG\s+CHK\b", "GH 9", regex=True)

    # Canonicalize CO/OC check variants
    s = s.str.replace(
        r"^(OC|CO)(\s*/\s*(OC|CO))?\s+CHK",
        "CO/OC CHK",
        regex=True,
    )

    # Fix compact RV forms and known weird value
    s = s.str.replace(r"\bRV\s*1\s*-\s*19-05-2015\b", "RV 1", regex=True)
    s = s.str.replace(r"\bRV1\b", "RV 1", regex=True)
    s = s.str.replace(r"\bRV2\b", "RV 2", regex=True)
    s = s.str.replace(r"\bRV3\b", "RV 3", regex=True)

    # Insert spaces between letters and digits
    # GH27 -> GH 27, OFS10 -> OFS 10, GH27R -> GH 27 R
    s = s.str.replace(r"(?<=[A-Z])(?=\d)", " ", regex=True)
    s = s.str.replace(r"(?<=\d)(?=[A-Z])", " ", regex=True)

    # Canonicalize lesson aliases after spacing normalization
    # GH 35 was originally GHT
    # GH 40 was originally BHT
    s = s.str.replace(r"\bGH\s+35\b", "GHT", regex=True)
    s = s.str.replace(r"\bGH\s+39\b", "BHT", regex=True)
    s = s.str.replace(r"\bGH\s+40\b", "BHT", regex=True)

    # Normalize compact suffix forms
    s = s.str.replace(r"\bGHTR\b", "GHT R", regex=True)
    s = s.str.replace(r"\bIFTR\b", "IFT R", regex=True)

    # Normalize "FM FL 1" -> "FM 1 FL", "FM FL 2" -> "FM 2 FL"
    s = s.str.replace(r"\bFM\s+FL\s+1\b", "FM 1 FL", regex=True)
    s = s.str.replace(r"\bFM\s+FL\s+2\b", "FM 2 FL", regex=True)

    # Normalize "NG FL 1" -> "NG 1 FL", "NG FL 2" -> "NG 2 FL"
    s = s.str.replace(r"\bNG\s+FL\s+1\b", "NG 1 FL", regex=True)
    s = s.str.replace(r"\bNG\s+FL\s+2\b", "NG 2 FL", regex=True)

    # Normalize "GH FL 1" -> "GH 1 FL", "GH FL 2" -> "GH 2 FL"
    s = s.str.replace(r"\bGH\s+FL\s+1\b", "GH 1 FL", regex=True)
    s = s.str.replace(r"\bGH\s+FL\s+2\b", "GH 2 FL", regex=True)

    # Normalize whitespace
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()

    out_df["LESSON_NUMBER"] = s
    return out_df

In [14]:
df = clean_lesson_numbers(df_bwc)

### Create `lesson_suffix` and `lesson_base` columns

In [15]:
def add_lesson_suffix(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add lesson_suffix from LESSON_NUMBER.

    lesson_suffix is:
      - 'R', 'FL', 'GS', or 'CX', 'OC' if the final token is one of these
      - <NA> otherwise
    """
    if "LESSON_NUMBER" not in df.columns:
        raise KeyError("Expected column 'LESSON_NUMBER' not found.")

    out_df = df.copy()

    s = out_df["LESSON_NUMBER"].astype("string").str.upper().str.strip()

    out_df["lesson_suffix"] = s.str.extract(r"\b(R|FL|GS|CX|OC)$", expand=False).astype(
        "string"
    )

    return out_df


def add_lesson_base(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add lesson_base from LESSON_NUMBER by removing final suffix token
    if it is one of R, FL, GS, CX, OC.
    """
    if "LESSON_NUMBER" not in df.columns:
        raise KeyError("Expected column 'LESSON_NUMBER' not found.")

    out_df = df.copy()

    s = out_df["LESSON_NUMBER"].astype("string").str.upper().str.strip()
    
    out_df["lesson_base"] = s.str.replace(
        r"(?:\s+(?:R|FL|GS|CX|OC))+$",
        "",
        regex=True,
    )

    return out_df

In [16]:
df = add_lesson_suffix(df)
df = add_lesson_base(df)

### Drop rows from BWC batches 150-159

There is no ground truth AWC stream data for BWC batches 150-159, hence we drop them.

In [17]:
df = df[df["bwc_batch"] >= 160].reset_index(drop=True)

### Drop rows with errors indicated by EMP_EVAL_COMMENTS

In [18]:
bef = len(df)
df = df[~df["EMP_EVAL_COMMENTS"].str.contains("Duplicated event", case=False, regex=False, na=False)]
df = df[~df["EMP_EVAL_COMMENTS"].str.contains("Double up gradesheet", case=False, regex=False, na=False)]
df = df[~df["EMP_EVAL_COMMENTS"].str.contains("Erroneous gradesheet", case=False, regex=False, na=False)]
df = df[~df["EMP_EVAL_COMMENTS"].str.contains("error gradesheet", case=False, regex=False, na=False)]
df = df[~df["EMP_EVAL_COMMENTS"].str.contains("error gs", case=False, regex=False, na=False)]
df = df[~df["EMP_EVAL_COMMENTS"].str.contains("wrong gs", case=False, regex=False, na=False)]
df = df[df["EMP_EVAL_COMMENTS"] != "ERROR"]
df = df[df["EMP_EVAL_COMMENTS"] != "error"]
aft = len(df)
print(f"{bef - aft} rows with errors indicated by EMP_EVAL_COMMENTS dropped. {aft} rows remain.")

13043 rows with errors indicated by EMP_EVAL_COMMENTS dropped. 840386 rows remain.


### Save clean data

In [19]:
# Sort df
df = df.sort_values(by=["bwc_batch", "STUDENT_ID", "LESSON_ACTUAL_START_DATE"]).reset_index(drop=True)

In [20]:
# Reorder columns for readability
first_cols = [
    "bwc_batch",
    "STUDENT_ID",
    "LESSON_NUMBER",
    "lesson_base",
    "lesson_suffix",
    "LESSON_ACTUAL_START_DATE",
    "LESSON_ACTUAL_END_DATE",
    "EMP_EVAL_SCORE",
    "EMP_EVAL_COMMENTS",
    "DUTY_STATUS_DESCRIPTION",
    "ADJUSTED_SCORE",
    "AVERAGE_SCORE",
    "OBJECTIVE_DESCRIPTION",
    "OBJECTIVE_RAW_SCORE",
    "OBJECTIVE_SCORE_WEIGHT",
    "OBJECTIVE_WEIGHTED_SCORE",
    "OBJECTIVE_CRITICAL",
    "OBJECTIVE_ACCEPTABLE_SCORE",
    "OBJECTIVE_ACCEPTABLE_SCORE_DSC",
]

remaining_cols = [col for col in df.columns if col not in first_cols]

df = df[first_cols + remaining_cols]

In [21]:
df.to_parquet("../data/02_interim/bwc_160-207.parquet", index=False)

## Filtering of clean data

In [22]:
df = pd.read_parquet("../data/02_interim/bwc_160-207.parquet")

### Keep only relevant lesson numbers

In [23]:
bef = len(df)
df= df[df["lesson_base"].isin(LESSONS_TO_KEEP)].reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows belonging to lesson numbers NOT in modules 1 to 3 dropped. {aft} rows remain.")

509595 rows belonging to lesson numbers NOT in modules 1 to 3 dropped. 330791 rows remain.


### Drop rows with lesson_suffix R, GS, FL

In [24]:
bef = len(df)
df = df[~df["lesson_suffix"].isin(["R", "GS", "FL"])].reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows with LESSON_NUMBER suffixed with R, GS, FL dropped. {aft} rows remain.")

11605 rows with LESSON_NUMBER suffixed with R, GS, FL dropped. 319186 rows remain.


### Drop rows with NG

If EMP_EVAL_COMMENTS contain ng or NG, OR lesson_suffix is GS, the sortie is considered to be NG i.e. not graded.

In [25]:
# Create helper ng boolean column
df["ng"] = (
    df["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\bNG\b", case=False, na=False)
)

# Check for ghosted solo in the comments, as some LESSON_NUMBERs have indicators of
# ghosted solo in the comments but have not been suffixed by GS.
mask = df["EMP_EVAL_COMMENTS"].astype("string").str.contains(r"Ghost", case=False, na=False)
df.loc[mask, "lesson_suffix"] = "GS"
needs_gs = mask & ~df["LESSON_NUMBER"].astype("string").str.endswith(" GS", na=False)
df.loc[needs_gs, "LESSON_NUMBER"] = df.loc[needs_gs, "LESSON_NUMBER"].astype("string") + " GS"

# Correct after manual inspection using excel
mask = (df["STUDENT_ID"] == "181GOHR") & (df["LESSON_NUMBER"] == "GH 22 GS")
df.loc[mask, "LESSON_NUMBER"] = "GH 22"
df.loc[mask, "lesson_suffix"] = pd.NA
mask = (df["STUDENT_ID"] == "199LIMH") & (df["LESSON_NUMBER"] == "GH 22 GS")
df.loc[mask, "LESSON_NUMBER"] = "GH 22"
df.loc[mask, "lesson_suffix"] = pd.NA
mask = (df["STUDENT_ID"] == "202WONGS") & (df["LESSON_NUMBER"] == "GH 25 GS")
df.loc[mask, "LESSON_NUMBER"] = "GH 25"
df.loc[mask, "lesson_suffix"] = pd.NA

# Ghosted solos are not graded.
mask = df["lesson_suffix"] == "GS"
df.loc[mask, "ng"] = True

# Drop rows with NG
bef = len(df)
df = df[~df["ng"]].reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows with NG indicators have been dropped. {aft} rows remain.")

# Remove helper column
df = df.drop(columns=["ng"])

28834 rows with NG indicators have been dropped. 290352 rows remain.


### Drop rows with DNCO

In [26]:
# Create boolean helper column
# Condition: If DUTY_STATUS_DESCRIPTION == DNCO, or if EMP_EVAL_COMMENTS contain DNCO
df["dnco"] = (
    df["DUTY_STATUS_DESCRIPTION"].astype("string").eq("DNCO")
    |
    df["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\bDNCO\b", case=False, na=False)
)
df["dnco"] = df["dnco"].fillna(False)

# Drop rows with DNCO
bef = len(df)
df = df[~df["dnco"]].reset_index(drop=True)
aft = len(df)
print(f"{bef - aft} rows with DNCO indicators have been dropped. {aft} rows remain.")

# Remove helper column
df = df.drop(columns=["dnco"])

2347 rows with DNCO indicators have been dropped. 288005 rows remain.


### Save

In [27]:
df.to_parquet("../data/02_interim/bwc_160-207_filtered_lessons.parquet", index=False)

In [28]:
df.to_csv("../data/02_interim/bwc_160-207_filtered_lessons.csv", index=False)

## Before pivoting

In [29]:
df = pd.read_parquet("../data/02_interim/bwc_160-207_filtered_lessons.parquet")

### Check for lesson_suffix

In [30]:
print(df["lesson_suffix"].value_counts(dropna=False))

print(df[df["lesson_suffix"] == "OC"]["LESSON_NUMBER"].unique())

df["LESSON_NUMBER"] = df["LESSON_NUMBER"].replace("GH 21 OC", "GH 21")

assert sorted(df["LESSON_NUMBER"].unique()) == sorted(LESSONS_TO_KEEP)

# remove helper columns
df = df.drop(columns=["lesson_base", "lesson_suffix"])


lesson_suffix
<NA>    287974
OC          31
Name: count, dtype: int64[pyarrow]
<ArrowStringArray>
['GH 21 OC']
Length: 1, dtype: string


### Create boolean `failed` column

In [31]:
# Condition: if EMP_EVAL_COMMENTS contains fail, failed, Fail, Failed
df["failed"] = (
    df["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\b(?:fail|failed)\b", case=False, na=False)
)

# Correct after manual inspection using excel
manual_failed_false = {
    ("194TNAVIN", "GH 5 R"),
    ("181NGW", "GH 29"),
    ("169ARVIND", "IF 3 R"),
    ("177AYUSH", "GH 4 R"),
    ("186TANJ", "GH 11"),
    ("204CKOH", "GH 14"),
    ("168MAHC", "GH 3"),
    ("176ZHUANGZ", "GH 8 R"),
    ("181NGY", "GH 6"),
    ("178LEEW", "GH 12"),
    ("198SATNAM", "GH 17"),
}

mask = list(zip(df["STUDENT_ID"], df["LESSON_NUMBER"]))
mask = pd.Series(mask, index=df.index).isin(manual_failed_false)

df.loc[mask, "failed"] = False

In [32]:
# Rearrange columns
first_cols = [
    "bwc_batch",
    "STUDENT_ID",
    "LESSON_NUMBER",
    "LESSON_ACTUAL_START_DATE",
    "LESSON_ACTUAL_END_DATE",
    "ADJUSTED_SCORE",
    "EMP_EVAL_SCORE",
    "EMP_EVAL_COMMENTS",
    "failed",
    "DUTY_STATUS_DESCRIPTION",
    "AVERAGE_SCORE",
    "OBJECTIVE_DESCRIPTION",
    "OBJECTIVE_RAW_SCORE",
    "OBJECTIVE_SCORE_WEIGHT",
    "OBJECTIVE_CRITICAL",
    "OBJECTIVE_ACCEPTABLE_SCORE",
    "OBJECTIVE_ACCEPTABLE_SCORE_DSC",
]

remaining_cols = [col for col in df.columns if col not in first_cols]

df = df[first_cols + remaining_cols]
df.head()

,bwc_batch,STUDENT_ID,LESSON_NUMBER,LESSON_ACTUAL_START_DATE,LESSON_ACTUAL_END_DATE,ADJUSTED_SCORE,EMP_EVAL_SCORE,EMP_EVAL_COMMENTS,failed,DUTY_STATUS_DESCRIPTION,...,EVALUATOR_ACKNOWLEDGED,EVALUATOR_ACKNOWLEDGED_DATE,EVAL_ACK_BY_EMPLOYEE_DS_ID,EMPLOYEE_ACKNOWLEDGED,EMPLOYEE_ACKNOWLEDGED_DATE,EMP_ACK_BY_EMPLOYEE_DS_ID,OBJECTIVE_ID,OBJECTIVE_EXPECTED_SCORE,OBJECTIVE_EXPECTED_SCORE_DSC,OBJECTIVE_COMMENTS
0,160,160CHANC,GH 1,2014-04-08 12:23:00,2014-04-08 13:44:00,<NA>,4.215324,[Nil]\r\n{Trainee checks and procedure was com...,False,DCO,...,True,2014-04-08 14:53:58,0289c8e2fe5e4cb48f4472272d0c9a2a,True,2014-04-15 19:44:55,1af7a5c3644d40528911d0dcb69e6450,SORTADM,2.0,2B,Trainee was prepared for the sortie.
1,160,160CHANC,GH 1,2014-04-08 12:23:00,2014-04-08 13:44:00,<NA>,4.215324,[Nil]\r\n{Trainee checks and procedure was com...,False,DCO,...,True,2014-04-08 14:53:58,0289c8e2fe5e4cb48f4472272d0c9a2a,True,2014-04-15 19:44:55,1af7a5c3644d40528911d0dcb69e6450,OPERAIRSYS,2.0,2B,Coming along. Required minor prompting
2,160,160CHANC,GH 1,2014-04-08 12:23:00,2014-04-08 13:44:00,<NA>,4.215324,[Nil]\r\n{Trainee checks and procedure was com...,False,DCO,...,True,2014-04-08 14:53:58,0289c8e2fe5e4cb48f4472272d0c9a2a,True,2014-04-15 19:44:55,1af7a5c3644d40528911d0dcb69e6450,AIRSTRAP,2.0,2B,Generally progressing fine. Coming along. Requ...
3,160,160CHANC,GH 1,2014-04-08 12:23:00,2014-04-08 13:44:00,<NA>,4.215324,[Nil]\r\n{Trainee checks and procedure was com...,False,DCO,...,True,2014-04-08 14:53:58,0289c8e2fe5e4cb48f4472272d0c9a2a,True,2014-04-15 19:44:55,1af7a5c3644d40528911d0dcb69e6450,AIRENGINES,2.0,2B,Coming along.
4,160,160CHANC,GH 1,2014-04-08 12:23:00,2014-04-08 13:44:00,<NA>,4.215324,[Nil]\r\n{Trainee checks and procedure was com...,False,DCO,...,True,2014-04-08 14:53:58,0289c8e2fe5e4cb48f4472272d0c9a2a,True,2014-04-15 19:44:55,1af7a5c3644d40528911d0dcb69e6450,COMSYSTEMS,2.0,2B,Trainee was able to communicate clearly and we...


In [33]:
print(sorted(df["bwc_batch"].unique()))

[np.int64(160), np.int64(161), np.int64(162), np.int64(163), np.int64(164), np.int64(165), np.int64(166), np.int64(167), np.int64(168), np.int64(169), np.int64(170), np.int64(171), np.int64(172), np.int64(173), np.int64(174), np.int64(175), np.int64(176), np.int64(177), np.int64(178), np.int64(179), np.int64(180), np.int64(181), np.int64(182), np.int64(183), np.int64(184), np.int64(185), np.int64(186), np.int64(187), np.int64(188), np.int64(189), np.int64(190), np.int64(191), np.int64(192), np.int64(193), np.int64(194), np.int64(195), np.int64(196), np.int64(197), np.int64(198), np.int64(199), np.int64(200), np.int64(201), np.int64(202), np.int64(203), np.int64(204), np.int64(205), np.int64(206), np.int64(207)]


### Keep only the first row of each LESSON_NUMBER for each STUDENT_ID

The data contains one row for each objective of a LESSON_NUMBER. All rows belonging to the same LESSON_NUMBER should have the same EMP_EVAL_SCORE. If duplicate EMP_EVAL_SCORE values exist, keep the first instance.

In [34]:
df_filtered = df.drop_duplicates(subset=["STUDENT_ID", "LESSON_NUMBER", "LESSON_ACTUAL_START_DATE"]).reset_index(drop=True)

### Drop duplicate combinations of STUDENT_ID, LESSON_NUMBER

At this point, each STUDENT_ID should only have ONE row per LESSON_NUMBER.

In [36]:
group_sizes = df_filtered.groupby(["STUDENT_ID", "LESSON_NUMBER"]).size()
bad_groups = group_sizes[group_sizes > 1]
assert bad_groups.empty, f"Found duplicate groups:\n{bad_groups}"

AssertionError: Found duplicate groups:
STUDENT_ID  LESSON_NUMBER
160HUANGZ   GH 29            2
            GH 31            2
160KHORM    GH 15            2
            GH 29            2
160LEEB     GH 19            2
                            ..
200YEOJ     GH 14            2
201HONT     GH 15            2
201LEEJ     GH 15            2
            GH 25            2
202PHUAC    GH 15            2
Length: 287, dtype: int64

In [37]:
bad_groups.sort_values(ascending=False).head(20)

STUDENT_ID  LESSON_NUMBER
168SOHJ     GH 17            4
176ENGJ     GHT              4
180KWEKW    GH 29            3
180ALIM     GH 31            3
167WANK     GH 29            3
178FUS      GH 29            3
168GANJ     GH 15            3
168LIMJ     GH 22            3
177YIPZ     GH 15            3
162ONGJ     GH 15            3
177AYUSH    GH 29            3
176FOOF     GH 31            3
160LEEB     GH 29            3
162BORNERH  GH 15            3
160LEES     GH 31            3
162KOHW     GH 15            3
178ANGC     IFT              2
178CHUAZ    GH 15            2
            GH 19            2
175GOHJ     GHT              2
dtype: int64

In [38]:
mask = (df_filtered["STUDENT_ID"] == "168SOHJ") & (df_filtered["LESSON_NUMBER"] == "GH 17")
print(df_filtered[mask]["LESSON_ACTUAL_START_DATE"].unique().tolist())
print(df_filtered[mask]["EMP_EVAL_COMMENTS"].unique().tolist())
print(df_filtered[mask]["EMP_EVAL_SCORE"].unique().tolist())

[Timestamp('2016-07-13 09:02:00'), Timestamp('2016-07-14 09:01:00'), Timestamp('2016-07-15 08:23:00'), Timestamp('2016-07-18 09:04:00')]
['Good circuit mechanics. However, needed a bit more headwork in wind effect and compensation technique during circuit flying.\r\nAirmanship / SA was fine.\r\n\r\nCleared solo.', 'Despite strong gusty winds being out of  trainee limits, able to execute several safe touch downs and landing.\r\n\r\nCan work the ground and fundamental lapses of incorrect avionics setting and inaccurate parameters for departure.\r\n\r\nCleared solo!', 'Need to work on error management and prioritisation. Debriefed to not be affect by errors and forget to conduct important checks or fly the aircraft parameters.\r\n\r\nSafe takeoff, rollers and landing. \r\nCleared solo.', 'No major problems.\r\nSafe take-off, rollers and landing.\r\n\r\nCleared solo!']
[3.8435493, 4.737143, 3.735357, 4.455357]


In [39]:
# Drop rows
bef = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(subset=["STUDENT_ID", "LESSON_NUMBER"])
aft = len(df_filtered)
print(f"{bef - aft} rows dropped")

305 rows dropped


In [40]:
group_sizes = df_filtered.groupby(["STUDENT_ID", "LESSON_NUMBER"]).size()
bad_groups = group_sizes[group_sizes > 1]
assert bad_groups.empty, f"Found duplicate groups:\n{bad_groups}"

### Save

In [41]:
df_filtered.to_parquet("../data/02_interim/bwc_160-207_one_row_per_lesson_number.parquet", index=False)
df_filtered.to_csv("../data/02_interim/bwc_160-207_one_row_per_lesson_number.csv", index=False)

## Pivot

In [42]:
df = pd.read_parquet("../data/02_interim/bwc_160-207_one_row_per_lesson_number.parquet")

# Fillna in ADJUSTED_SCORE with EMP_EVAL_SCORE 
df["ADJUSTED_SCORE"] = df["ADJUSTED_SCORE"].fillna(df["EMP_EVAL_SCORE"])

# Sanity check: Ensure that there is only one row per STUDENT_ID and LESSON_NUMBER
group_sizes = df.groupby(["STUDENT_ID", "LESSON_NUMBER"]).size()
bad_groups = group_sizes[group_sizes > 1]
assert bad_groups.empty, f"Found duplicate groups:\n{bad_groups}"
assert df["failed"].isnull().sum() == 0
assert df["EMP_EVAL_SCORE"].isnull().sum() == 0
assert df["ADJUSTED_SCORE"].isnull().sum() == 0

In [43]:
# Pivot table
wide = (
    df.pivot_table(
        index="STUDENT_ID",
        columns="LESSON_NUMBER",
        values=["EMP_EVAL_SCORE", "ADJUSTED_SCORE", "failed"],
        aggfunc="first",
    )
)

# Reorder columns so lesson order is preserved, and within each lesson: score, adjscore, failed
ordered_cols = [
    (value_col, lesson)
    for lesson in LESSONS_TO_KEEP
    for value_col in ["EMP_EVAL_SCORE", "ADJUSTED_SCORE", "failed"]
    if (value_col, lesson) in wide.columns
]
wide = wide.reindex(columns=ordered_cols)

# Flatten multiindex columns
name_map = {
    "EMP_EVAL_SCORE": "score",
    "ADJUSTED_SCORE": "adjscore",
    "failed": "failed",
}
wide.columns = [
    f"{lesson}_{name_map[value_col]}"
    for value_col, lesson in wide.columns
]
wide = wide.reset_index()
wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 739 entries, 0 to 738
Columns: 112 entries, STUDENT_ID to IFT_failed
dtypes: Float64(74), boolean(37), string(1)
memory usage: 545.4 KB


In [44]:
# add missingness indicators based on *_score
for lesson in LESSONS_TO_KEEP:
    score_col = f"{lesson}_score"
    missing_col = f"{lesson}_missing"

    if score_col in wide.columns:
        wide[missing_col] = wide[score_col].isna().astype("boolean")
        
# final column order
final_cols = ["STUDENT_ID"]
for lesson in LESSONS_TO_KEEP:
    for suffix in ["score", "adjscore", "failed", "missing"]:
        col = f"{lesson}_{suffix}"
        if col in wide.columns:
            final_cols.append(col)

wide = wide[final_cols]
wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 739 entries, 0 to 738
Columns: 149 entries, STUDENT_ID to IFT_missing
dtypes: Float64(74), boolean(74), string(1)
memory usage: 598.8 KB


In [45]:
wide

,STUDENT_ID,GH 1_score,GH 1_adjscore,GH 1_failed,GH 1_missing,GH 2_score,GH 2_adjscore,GH 2_failed,GH 2_missing,GH 3_score,...,IF 5_failed,IF 5_missing,IF 6_score,IF 6_adjscore,IF 6_failed,IF 6_missing,IFT_score,IFT_adjscore,IFT_failed,IFT_missing
0,160CHANC,4.215324,4.215324,False,False,3.937536,3.937536,False,False,4.0,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
1,160CHANG,3.926446,3.926446,False,False,3.746286,3.746286,False,False,4.094832,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
2,160CHOOC,4.243429,4.243429,False,False,4.37966,4.37966,False,False,4.674136,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
3,160HOS,4.556401,4.556401,False,False,4.476572,4.476572,False,False,4.458939,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
4,160HOWW,4.936332,4.936332,False,False,4.061821,4.061821,False,False,3.903645,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,207LUH,5.420786,5.420786,False,False,<NA>,<NA>,<NA>,True,4.912946,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
735,207SEAHI,<NA>,<NA>,<NA>,True,5.342156,5.342156,False,False,4.52863,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
736,207SOHY,4.87172,4.87172,False,False,4.977205,4.977205,False,False,4.657893,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True
737,207TANK,4.994169,4.994169,False,False,5.210692,5.210692,False,False,<NA>,...,<NA>,True,<NA>,<NA>,<NA>,True,<NA>,<NA>,<NA>,True


## Append ground truth labels

### Get ground truth labels

In [46]:
gt_160_207_df = pd.read_parquet("../data/labels/my_BWC160t207_Status_as_of_end_Feb2026.parquet")

#### Extract

In [47]:
# BWC batches 201 to 207
gt_201_207_df = pd.read_excel("../data/labels/BWC201t207_Status_as_of_end_Feb2026.xlsx")

In [48]:
# BWC batches 160 to 200
gt_160_200_df = pd.read_csv("../data/labels/BWC_AWCstream_NON_EMPTY.csv")
gt_160_200_df = gt_160_200_df[["STUDENT_ID", "AWC_stream", "av_creoc"]]

# Follow format of BWC batches 201 to 207
Stream_Unit_dict = {
    1: "FWC",
    2: "RWC",
    3: "TWC",
    4: "NFTC",
    5: "ITAF",
    6: "IERW",
    7: "SUPT"
}
gt_160_200_df["Stream_Unit"] = gt_160_200_df["AWC_stream"].map(Stream_Unit_dict)

gt_160_200_df.loc[gt_160_200_df["Stream_Unit"].isin(["FWC", "NFTC", "ITAF", "SUPT"]), "Stream_Group"] = "Fighter"
gt_160_200_df.loc[gt_160_200_df["Stream_Unit"].isin(["RWC", "IERW"]), "Stream_Group"] = "Heli"
gt_160_200_df.loc[gt_160_200_df["Stream_Unit"].isin(["TWC", "Stream_Group"]), "Stream_Group"] = "Transport"

gt_160_200_df["BWC_Status"] = "Pass"
gt_160_200_df.loc[gt_160_200_df["Stream_Unit"] == "SUPT", "BWC_Status"] = "DNF(SUPT)"

gt_160_200_df["Batch"] = (
    gt_160_200_df["STUDENT_ID"]
    .astype("string")
    .str.extract(r"^(\d+)", expand=False)
    .astype("Int64")
)

gt_160_200_df = gt_160_200_df.rename(columns={"STUDENT_ID": "STUDENT_ID_ORIG"})
gt_160_200_df = gt_160_200_df[gt_201_207_df.columns]

gt_160_200_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   STUDENT_ID_ORIG  358 non-null    str  
 1   Batch            358 non-null    Int64
 2   BWC_Status       358 non-null    str  
 3   Stream_Group     358 non-null    str  
 4   Stream_Unit      358 non-null    str  
dtypes: Int64(1), str(4)
memory usage: 22.0 KB


In [49]:
# Merge batches 160 to 207 into a singular df
gt_160_207_df = pd.concat([gt_160_200_df, gt_201_207_df], ignore_index=True)
gt_160_207_df = gt_160_207_df.rename(columns={"STUDENT_ID_ORIG": "STUDENT_ID"})

In [50]:
# Enforce schema
string_cols = [
    "STUDENT_ID",
    "BWC_Status",
    "Stream_Group",
    "Stream_Unit",
]
int_cols = ["Batch"]

gt_160_207_df[int_cols] = gt_160_207_df[int_cols].apply(pd.to_numeric, errors="coerce").astype("Int64")
gt_160_207_df[string_cols] = gt_160_207_df[string_cols].astype("string")

In [51]:
gt_160_207_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 458 entries, 0 to 457
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   STUDENT_ID    458 non-null    string
 1   Batch         458 non-null    Int64 
 2   BWC_Status    458 non-null    string
 3   Stream_Group  400 non-null    string
 4   Stream_Unit   400 non-null    string
dtypes: Int64(1), string(4)
memory usage: 27.8 KB


In [52]:
gt_160_207_df.to_csv("../data/labels/my_BWC160t207_Status_as_of_end_Feb2026.csv", index=False)
gt_160_207_df.to_parquet("../data/labels/my_BWC160t207_Status_as_of_end_Feb2026.parquet", index=False)

### Merge

In [53]:
merged = wide.merge(
    gt_160_207_df,
    on="STUDENT_ID",
    how="left",
)

In [54]:
# Fill NaN values
batch_from_student_id = (
    merged["STUDENT_ID"]
    .astype("string")
    .str.extract(r"^(\d+)", expand=False)
    .astype("Int64")
)
merged["Batch"] = merged["Batch"].astype("Int64").fillna(batch_from_student_id)
merged["BWC_Status"] = merged["BWC_Status"].fillna("Fail")

In [55]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 739 entries, 0 to 738
Columns: 153 entries, STUDENT_ID to Stream_Unit
dtypes: Float64(74), Int64(1), boolean(74), string(4)
memory usage: 629.8 KB


In [56]:
merged

,STUDENT_ID,GH 1_score,GH 1_adjscore,GH 1_failed,GH 1_missing,GH 2_score,GH 2_adjscore,GH 2_failed,GH 2_missing,GH 3_score,...,IF 6_failed,IF 6_missing,IFT_score,IFT_adjscore,IFT_failed,IFT_missing,Batch,BWC_Status,Stream_Group,Stream_Unit
0,160CHANC,4.215324,4.215324,False,False,3.937536,3.937536,False,False,4.0,...,<NA>,True,<NA>,<NA>,<NA>,True,160,Fail,<NA>,<NA>
1,160CHANG,3.926446,3.926446,False,False,3.746286,3.746286,False,False,4.094832,...,<NA>,True,<NA>,<NA>,<NA>,True,160,Fail,<NA>,<NA>
2,160CHOOC,4.243429,4.243429,False,False,4.37966,4.37966,False,False,4.674136,...,<NA>,True,<NA>,<NA>,<NA>,True,160,DNF(SUPT),Fighter,SUPT
3,160HOS,4.556401,4.556401,False,False,4.476572,4.476572,False,False,4.458939,...,<NA>,True,<NA>,<NA>,<NA>,True,160,DNF(SUPT),Fighter,SUPT
4,160HOWW,4.936332,4.936332,False,False,4.061821,4.061821,False,False,3.903645,...,<NA>,True,<NA>,<NA>,<NA>,True,160,Fail,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,207LUH,5.420786,5.420786,False,False,<NA>,<NA>,<NA>,True,4.912946,...,<NA>,True,<NA>,<NA>,<NA>,True,207,Ongoing,<NA>,<NA>
735,207SEAHI,<NA>,<NA>,<NA>,True,5.342156,5.342156,False,False,4.52863,...,<NA>,True,<NA>,<NA>,<NA>,True,207,Ongoing,<NA>,<NA>
736,207SOHY,4.87172,4.87172,False,False,4.977205,4.977205,False,False,4.657893,...,<NA>,True,<NA>,<NA>,<NA>,True,207,Ongoing,<NA>,<NA>
737,207TANK,4.994169,4.994169,False,False,5.210692,5.210692,False,False,<NA>,...,<NA>,True,<NA>,<NA>,<NA>,True,207,Ongoing,<NA>,<NA>


In [57]:
for col in ['BWC_Status','Stream_Group','Stream_Unit']:
    print(col, merged[col].value_counts(dropna=False).to_dict())

BWC_Status {'Pass': 351, 'Fail': 301, 'DNF(SUPT)': 49, 'Ongoing': 30, 'DNF(UGPS)': 4, 'DNF(Disciplinary)': 3, 'DNF(medrollBWC205)': 1}
Stream_Group {<NA>: 339, 'Fighter': 201, 'Heli': 136, 'Transport': 63}
Stream_Unit {<NA>: 339, 'RWC': 125, 'FWC': 100, 'TWC': 63, 'SUPT': 49, 'NFTC': 32, 'ITAF': 20, 'IERW': 11}


### Add clean target labels

In [58]:
# target_binary: pass or fail bwc
# - Pass: `["Pass", "DNF(SUPT)"]`
# - Fail: ` ["Fail"]`
# - Others: `["Ongoing", "DNF(Disciplinary)", "Suspended(PsyReview)", "Airsick", "DNF(medrollBWCXXX i.e. rollover to future BWC course xxx due to medical reasons)", "DNF(UGPS i.e. trainee has disrupted pilot training to pursue undergraduate studies)"]`  
merged["target_binary"] = pd.NA
merged.loc[merged["BWC_Status"].isin(["Pass", "DNF(SUPT)"]), "target_binary"] = 1
merged.loc[merged["BWC_Status"].isin(["Fail"]), "target_binary"] = 0
merged["target_binary"] = merged["target_binary"].astype("Int64")

# Drop STUDENT_IDs that did not clearly pass or fail BWC.
merged = merged.dropna(subset="target_binary").reset_index(drop=True)
merged["target_binary"].value_counts(dropna=False)

target_binary
1    400
0    301
Name: count, dtype: Int64

In [59]:
# target_multiclass: 0 for fail bwc, 1 for fighter, 2 for transport, 3 for heli
merged["target_multiclass"] = 0

# Pass BWC
merged.loc[merged["Stream_Group"] == "Fighter", "target_multiclass"] = 1
merged.loc[merged["Stream_Group"] == "Transport", "target_multiclass"] = 2
merged.loc[merged["Stream_Group"] == "Heli", "target_multiclass"] = 3

merged["target_multiclass"] = merged["target_multiclass"].astype("Int64")
merged["target_multiclass"].value_counts(dropna=False)

target_multiclass
0    301
1    201
3    136
2     63
Name: count, dtype: Int64

In [60]:
# target_fighter: 0 for fail BWC, transport, heli; 1 for fighter
merged["target_fighter"] = 0
merged.loc[merged["target_multiclass"] == 1, "target_fighter"] = 1

merged["target_fighter"] = merged["target_fighter"].astype("Int64")
merged["target_fighter"].value_counts(dropna=False)

target_fighter
0    500
1    201
Name: count, dtype: Int64

In [61]:
merged

,STUDENT_ID,GH 1_score,GH 1_adjscore,GH 1_failed,GH 1_missing,GH 2_score,GH 2_adjscore,GH 2_failed,GH 2_missing,GH 3_score,...,IFT_adjscore,IFT_failed,IFT_missing,Batch,BWC_Status,Stream_Group,Stream_Unit,target_binary,target_multiclass,target_fighter
0,160CHANC,4.215324,4.215324,False,False,3.937536,3.937536,False,False,4.0,...,<NA>,<NA>,True,160,Fail,<NA>,<NA>,0,0,0
1,160CHANG,3.926446,3.926446,False,False,3.746286,3.746286,False,False,4.094832,...,<NA>,<NA>,True,160,Fail,<NA>,<NA>,0,0,0
2,160CHOOC,4.243429,4.243429,False,False,4.37966,4.37966,False,False,4.674136,...,<NA>,<NA>,True,160,DNF(SUPT),Fighter,SUPT,1,1,1
3,160HOS,4.556401,4.556401,False,False,4.476572,4.476572,False,False,4.458939,...,<NA>,<NA>,True,160,DNF(SUPT),Fighter,SUPT,1,1,1
4,160HOWW,4.936332,4.936332,False,False,4.061821,4.061821,False,False,3.903645,...,<NA>,<NA>,True,160,Fail,<NA>,<NA>,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696,205LUIL,4.924317,4.924317,False,False,<NA>,<NA>,<NA>,True,4.917947,...,4.0,False,False,205,Pass,Fighter,FWC,1,1,1
697,205TANW,<NA>,<NA>,<NA>,True,5.793943,5.793943,False,False,<NA>,...,4.258857,False,False,205,Pass,Heli,RWC,1,3,0
698,205TJOENGY,<NA>,<NA>,<NA>,True,4.908005,4.908005,False,False,4.98627,...,1.0,True,False,205,Pass,Transport,TWC,1,2,0
699,205YEOC,5.97435,5.97435,False,False,6.025611,6.025611,False,False,5.163366,...,<NA>,<NA>,True,205,DNF(SUPT),Fighter,SUPT,1,1,1


### Save

In [62]:
merged.to_parquet("../data/03_clean/bwc_160_207.parquet", index=False)
merged.to_csv("../data/03_clean/bwc_160_207.csv", index=False)